# 02 - Arbitrary-length block 생성과 KV cache 회계

**학습 목표**: EOS가 나올 때까지 block을 추가하고, 완료 prefix의 KV를 다시 계산하지 않는 toy sampler를 구현합니다. 실제 Transformer KV tensor는 사용하지 않습니다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리만 사용합니다.

In [ ]:
MASK = '[MASK]'
EOS = '[EOS]'
reference = 'blocks can extend beyond a fixed training window safely'.split() + [EOS]

def denoise_block(reference, start, block_size):
    state = [MASK] * block_size
    # confidence 순서를 흉내 내되 최종적으로 reference를 복원합니다.
    order = list(range(0, block_size, 2)) + list(range(1, block_size, 2))
    evaluations = 0
    for position in order:
        absolute = start + position
        if absolute >= len(reference):
            break
        state[position] = reference[absolute]
        evaluations += 1
    return [x for x in state if x != MASK], evaluations


In [ ]:
def generate(reference, block_size, max_blocks=10):
    output, kv_cache = [], {}
    denoise_evaluations = 0
    for block_id in range(max_blocks):
        block, nfe = denoise_block(reference, len(output), block_size)
        denoise_evaluations += nfe
        for token in block:
            output.append(token)
            kv_cache[len(output) - 1] = f'cached({token})'
            if token == EOS:
                return output, kv_cache, denoise_evaluations
    return output, kv_cache, denoise_evaluations

output, cache, nfe = generate(reference, block_size=3)
print('output:', ' '.join(output))
print('cached prefix positions:', len(cache), 'toy denoise evaluations:', nfe)
assert output == reference
assert len(cache) == len(output)

실제 BD3-LM은 block 안 여러 위치를 한 forward에서 복원할 수 있고 NFE가 token 수 이하가 되도록 efficient masked sampler를 사용합니다. 여기서는 cache 수명과 EOS 기반 가변 길이만 분리해 보여줍니다.